# URECA: Random Forest Prediction of Fluorophore Photophysical Properties

**Project:** NTU URECA (self-directed), supervised by Prof Liu

This is a single notebook covering the whole pipeline: setup, EDA, featurization, modeling, analysis, and write-up. Everything is built in order, in this one file.

---

## Goal

We're predicting four photophysical properties of organic fluorophores (chromophores) from molecular structure (SMILES string):

| Target | Column in raw data | Notes |
|---|---|---|
| Absorption max | `Absorption max (nm)` | |
| Emission max | `Emission max (nm)` | |
| Extinction coefficient | `log(e/mol-1 dm3 cm-1)` | already log10-scaled in the raw data |
| Quantum yield (Φ_F) | `Quantum yield` | bounded 0–1, expected to be the hardest target |

**Model:** Random Forest, as the first model.

**Key method:** train/test split by **Murcko scaffold** (via RDKit), not randomly. A random split can let near-identical molecules appear on both sides, so the model can do well just by memorizing similar structures. Comparing performance on a scaffold split vs. a random split shows how well the model actually generalizes to new chemical structures — this gap is a potential result worth reporting.

## Notebook sections

1. **Setup & data loading**
2. EDA — distributions, missingness, solvent effects, correlations
3. Featurization + scaffold split
4. RF tuning
5. Feature importance + scaffold-vs-random gap analysis
6. Error analysis
7. Write-up

## Dataset

`chromophore_data.csv`, in this same folder. 20,836 rows, one row per (chromophore, solvent) measurement. Columns: SMILES, solvent, the four target properties, plus lifetime, FWHM, molecular weight, and reference.

Two things to note: not every row has all four targets (missingness varies by target), and the same molecule can appear multiple times under different solvents. Both are covered in the EDA section below.

## 1. Imports

`pandas`/`numpy` for data handling, `matplotlib`/`seaborn` for plots, `rdkit` for chemistry (reading SMILES, scaffolds, descriptors), `scikit-learn` for the model and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')  # RDKit likes to complain about every weird SMILES - silence that for now

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

print('rdkit, sklearn, pandas all imported OK')

## 2. Load the raw dataset

Read the CSV from this folder. `df_raw` stays untouched as a backup; all work happens on `df`.

In [ ]:
DATA_PATH = 'chromophore_data.csv'

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
df = df_raw.copy()  # work on a copy, keep df_raw untouched as a backup

print(f'Loaded {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

## 3. Rename columns

The raw column names (e.g. `Absorption max (nm)`) are long to type repeatedly, so we rename them to short codes. `df_raw` keeps the original names.

In [ ]:
RENAME_MAP = {
    'Chromophore': 'smiles',
    'Solvent': 'solvent',
    'Absorption max (nm)': 'abs_max',
    'Emission max (nm)': 'emi_max',
    'Quantum yield': 'qy',
    'log(e/mol-1 dm3 cm-1)': 'log_ext_coeff',
    'Molecular weight (g mol-1)': 'mol_weight',
}

df = df.rename(columns=RENAME_MAP)

# these two lists get reused constantly from here on, so let's name them once
TARGET_COLS = ['abs_max', 'emi_max', 'log_ext_coeff', 'qy']
ID_COLS = ['smiles', 'solvent']

df[ID_COLS + TARGET_COLS].head()

## 4. Sanity checks

Three checks before doing anything else: how much data each target actually has, whether RDKit can parse every SMILES string, and how many unique molecules we have (since the same molecule appears under multiple solvents).

In [ ]:
print('Non-null counts per target (out of {:,} rows):'.format(len(df)))
print(df[TARGET_COLS].notna().sum())
print()
print('Unique SMILES (molecules):', df['smiles'].nunique())
print('Unique solvents:', df['solvent'].nunique())

In [ ]:
def parses_ok(smi):
    # if RDKit can't parse a SMILES, Chem.MolFromSmiles just quietly returns None
    try:
        return Chem.MolFromSmiles(smi) is not None
    except Exception:
        return False

parse_mask = df['smiles'].apply(parses_ok)
print(f'SMILES that RDKit can parse: {parse_mask.sum():,} / {len(df):,} ({parse_mask.mean():.1%})')

if (~parse_mask).any():
    print('\nExample unparseable SMILES:')
    display(df.loc[~parse_mask, ['smiles']].head())

## 5. Exploratory Data Analysis

Go through the four targets one at a time: distribution, missingness, outliers, then a short findings note before moving to the next. After all four, look at cross-target relationships (e.g. Stokes shift = emission − absorption) and solvent effects.

### 5.1 Absorption max (nm)

In [ ]:
# how many rows actually have this value, and what does it look like on average?
print(f"Missing: {df['abs_max'].isna().sum():,} / {len(df):,} rows ({df['abs_max'].isna().mean():.1%})")
df['abs_max'].describe()

In [ ]:
# histogram for the overall shape, boxplot to spot outliers at a glance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['abs_max'].dropna(), bins=60, ax=axes[0], color='steelblue')
axes[0].set_title('Absorption max — distribution')
axes[0].set_xlabel('Absorption max (nm)')

sns.boxplot(x=df['abs_max'].dropna(), ax=axes[1], color='steelblue')
axes[1].set_title('Absorption max — boxplot (outlier check)')
axes[1].set_xlabel('Absorption max (nm)')

plt.tight_layout()
plt.show()

In [ ]:
# real UV-Vis-NIR absorption lives roughly in the 200-1000 nm window,
# so anything outside that is worth eyeballing - could be a typo in the source paper
low = df[df['abs_max'] < 200][['smiles', 'solvent', 'abs_max']]
high = df[df['abs_max'] > 1000][['smiles', 'solvent', 'abs_max']]

print(f'Below 200 nm: {len(low)} rows')
print(f'Above 1000 nm: {len(high)} rows')
display(pd.concat([low, high]))

**Findings — Absorption max:**

- **Coverage:** 17,703 / 20,836 rows (~85%) — the best-covered target.
- **Shape:** 127.7–1055.0 nm, median 397 nm, right-skewed (skew ≈ 1.25). Most values fall in the near-UV/visible range (350–480 nm is the middle 50%), with a long tail toward the NIR.
- **Outliers checked:** 21 rows below 200 nm and 1 above 1000 nm. All are explainable — the sub-200nm rows are small gas-phase molecules (methane, ethane, propane, etc.) that genuinely absorb in the far-UV; the >1000nm row is a large conjugated dye. None are data errors, so nothing gets dropped.
- **No other issues** — no zeros or negatives.
- **For modeling:** Random Forest handles skew and outliers fine, so no transform is needed for this target.

Next: Emission max.

### 5.2 Emission max (nm)

Same checks as above, plus one extra: emission should physically occur at a longer wavelength than absorption (Stokes shift), so we can use that as a sanity check.

In [ ]:
print(f"Missing: {df['emi_max'].isna().sum():,} / {len(df):,} rows ({df['emi_max'].isna().mean():.1%})")
df['emi_max'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['emi_max'].dropna(), bins=60, ax=axes[0], color='darkorange')
axes[0].set_title('Emission max — distribution')
axes[0].set_xlabel('Emission max (nm)')

sns.boxplot(x=df['emi_max'].dropna(), ax=axes[1], color='darkorange')
axes[1].set_title('Emission max — boxplot (outlier check)')
axes[1].set_xlabel('Emission max (nm)')

plt.tight_layout()
plt.show()

In [ ]:
# physics check: a molecule loses a bit of energy before it emits, so emission
# wavelength should always be >= absorption wavelength (positive Stokes shift).
# if we see the opposite, that's either unusual photophysics or a data issue worth a look.
both = df.dropna(subset=['emi_max', 'abs_max']).copy()
both['stokes_shift'] = both['emi_max'] - both['abs_max']

neg_shift = both[both['stokes_shift'] < 0].sort_values('stokes_shift')
print(f"Rows with emission < absorption (negative Stokes shift): {len(neg_shift)} / {len(both)}")
display(neg_shift[['smiles', 'solvent', 'abs_max', 'emi_max', 'stokes_shift']].head(10))

**Findings — Emission max:**

- **Coverage:** 18,851 / 20,836 rows (~90%) — better covered than absorption.
- **Shape:** 247–1050 nm, median 488 nm, mildly right-skewed (skew ≈ 0.66, less skewed than absorption). Consistent with the Stokes shift pushing emission to longer wavelengths than absorption.
- **Outliers:** only 1 row above 1000 nm (a long conjugated dye, plausible), none below 200 nm.
- **Stokes shift check:** 17 / 16,422 rows (~0.1%) have emission at a shorter wavelength than absorption, which shouldn't normally happen. 15 are small (≤15 nm), likely measurement noise. 2 are larger (-57, -55 nm, both BODIPY-dimer structures in toluene) — worth a second look later, but too few rows to drop.
- **For modeling:** cleaner than absorption max, no transform needed.

Next: log(extinction coefficient).

### 5.3 log(extinction coefficient)

Already log10-scaled in the raw data. This target has the most missingness of the four, so coverage matters here more than for the others.

In [ ]:
print(f"Missing: {df['log_ext_coeff'].isna().sum():,} / {len(df):,} rows ({df['log_ext_coeff'].isna().mean():.1%})")
df['log_ext_coeff'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['log_ext_coeff'].dropna(), bins=60, ax=axes[0], color='seagreen')
axes[0].set_title('log(extinction coefficient) — distribution')
axes[0].set_xlabel('log10(ε / mol⁻¹ dm³ cm⁻¹)')

sns.boxplot(x=df['log_ext_coeff'].dropna(), ax=axes[1], color='seagreen')
axes[1].set_title('log(extinction coefficient) — boxplot (outlier check)')
axes[1].set_xlabel('log10(ε / mol⁻¹ dm³ cm⁻¹)')

plt.tight_layout()
plt.show()

In [ ]:
# real molar extinction coefficients for organic dyes top out around 10^5-10^6
# (log10 ~ 5-6). anything above that is a candidate data error, not real chemistry.
high = df[df['log_ext_coeff'] > 6][['smiles', 'solvent', 'log_ext_coeff', 'mol_weight', 'Reference']]
print(f'Rows above log10(ε) = 6: {len(high)}')
display(high)

In [ ]:
# two of those four rows (log10(e) = 10.4 and 9.0) are from the same paper and the
# same family of near-identical molecules, where every other row sits around 5.5-6.0.
# that's a strong sign those two specific values are data-entry errors, not real values.
# fixing it directly on df - df_raw still has the original numbers if we ever need them.
error_mask = df['log_ext_coeff'] > 7
print(f'Setting {error_mask.sum()} likely data-error rows to NaN in log_ext_coeff')
df.loc[error_mask, 'log_ext_coeff'] = np.nan

**Findings — log(extinction coefficient):**

- **Coverage:** only 8,300 / 20,836 rows (~40%) — by far the most missing of the four targets. Fewer papers report this than absorption/emission.
- **Shape:** 1.08–10.4 before cleaning, median 4.46, left-skewed (skew ≈ -1.40, the only target skewed in this direction). Most values sit between 4.1 and 4.7 (log10 scale), i.e. ε roughly 10⁴–10⁵.
- **Outliers:** 4 rows above log10(ε) = 6. Two (10.4 and 9.0) are from the same paper and the same family of near-identical molecules, where every comparable structure in that paper sits around 5.5–6.0 — these two are almost certainly data-entry errors, not real chemistry (a molar extinction coefficient of 10⁹–10¹⁰ is physically implausible for an organic dye). Set to NaN rather than left in. The other two (6.02, 6.11) are plausible for strongly absorbing dyes and were kept.
- **For modeling:** low coverage means this target will have the smallest usable training set of the four — worth keeping in mind when comparing scaffold-split performance across targets later, since a smaller training set makes the scaffold-split gap harder to measure precisely.

Next: Quantum yield.

### 5.4 Quantum yield (Φ_F)

Bounded between 0 and 1 by definition, so the main things to check are whether the data actually respects that bound, and what the shape looks like near the edges.

In [ ]:
print(f"Missing: {df['qy'].isna().sum():,} / {len(df):,} rows ({df['qy'].isna().mean():.1%})")
df['qy'].describe()

In [ ]:
# quantum yield is bounded [0,1] by definition, so confirm the data actually respects that
qy = df['qy']
print('Values < 0:', (qy < 0).sum())
print('Values > 1:', (qy > 1).sum())
print('Values == 0:', (qy == 0).sum())
print('Values < 0.01:', (qy < 0.01).sum())
print('Values > 0.99:', (qy > 0.99).sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['qy'].dropna(), bins=50, ax=axes[0], color='mediumorchid')
axes[0].set_title('Quantum yield — distribution')
axes[0].set_xlabel('Quantum yield (Φ_F)')

sns.boxplot(x=df['qy'].dropna(), ax=axes[1], color='mediumorchid')
axes[1].set_title('Quantum yield — boxplot')
axes[1].set_xlabel('Quantum yield (Φ_F)')

plt.tight_layout()
plt.show()

**Findings — Quantum yield:**

- **Coverage:** 13,978 / 20,836 rows (~67%).
- **All values are within [0, 1]** — no cleaning needed on that front.
- **Shape is the real issue:** median 0.27, mean 0.35, mildly right-skewed (skew ≈ 0.57), but the skew number undersells it. 150 rows are exactly 0, and 1,273 (~9%) are below 0.01 — a sharp spike of non-emissive or near-non-emissive molecules at the low end. There's a separate smaller pile near 1 (148 rows above 0.99). So the real shape is closer to a U/bimodal distribution with a big mass at zero, not a smooth skewed curve.
- **Why this is the hardest target:** a large zero-inflated spike like this is hard for any regressor, RF included, to fit well — it has to learn a sharp discontinuity (near-zero vs. everything else) rather than a smooth function of molecular features. This matches the expectation going in.
- **For modeling:** worth trying a couple of specific strategies when we get there — e.g. a classification step (emissive vs. non-emissive) before regression, or just accepting a weaker R² here than for the wavelength targets and reporting it as expected, not a bug.

That's all four targets. Summary review next.

### 5.5 Data cleaning & target strategy — decisions going into modeling

Summary of what we decided after reviewing all four targets, before moving on to featurization:

- **Model structure:** four separate Random Forest models, one per target. No multi-output model.
- **Absorption max, Emission max:** no transform. Skew is mild (1.25, 0.66) and all checked outliers turned out to be real chemistry, not errors. Train on raw nm values.
- **log(extinction coefficient):** already log10-scaled in the raw data. Two rows (10.4, 9.0) were corrected to NaN as data-entry errors (Section 5.3). No further transform on top of the existing log scale.
- **Quantum yield:** train the baseline RF on the raw [0,1] values, no transform, even though the distribution is zero-inflated and this target is expected to perform worse. This keeps the core deliverable simple (one RF per target, same recipe each time) and gives an honest baseline number to report.
  - **Follow-up experiment (later, time permitting):** once the baseline for all four targets is done, revisit quantum yield specifically with an arcsine-sqrt transform and/or a two-stage classify-then-regress approach, and compare against the baseline empirically rather than assuming it's better.
- **Outlier policy in general:** no capping/winsorizing. Random Forest isolates extreme points into their own leaves rather than letting them distort a global fit the way a linear model would, so outliers we've confirmed as real don't need special handling.
- **Derived quantities (e.g. Stokes shift) are for EDA/error-analysis only** — they use a second target as input, so they can't be used as model features without leaking the thing we're trying to predict.

Next: featurization (turning SMILES into model inputs) and the scaffold split.

## 6. Data Cleaning

One more structural issue to handle before featurization: the same (molecule, solvent) pair sometimes appears as more than one row, because different papers measured the same compound independently. Checking how common this is and what to do about it.

In [ ]:
# same (smiles, solvent) pair measured more than once, usually by different papers
dup_mask = df.duplicated(subset=['smiles', 'solvent'], keep=False)
n_dup_rows = dup_mask.sum()
n_dup_groups = df[dup_mask].groupby(['smiles', 'solvent']).ngroups

print(f'Rows involved in a duplicate (smiles, solvent) pair: {n_dup_rows} / {len(df)}')
print(f'Number of distinct duplicated groups: {n_dup_groups}')

# how much do the repeated measurements actually disagree with each other?
spread = (df[dup_mask].groupby(['smiles', 'solvent'])['abs_max']
          .agg(lambda x: x.dropna().max() - x.dropna().min() if x.notna().sum() > 1 else np.nan))
print(f"\nAbsorption max spread within duplicate groups (where measured twice):")
print(spread.dropna().describe())
print(f"Groups disagreeing by >10 nm: {(spread.dropna() > 10).sum()} / {spread.dropna().shape[0]}")

In [ ]:
# collapse duplicate (smiles, solvent) rows into one, taking the median of each target.
# median is robust to the occasional one-off disagreement between papers without needing
# to pick a "correct" paper. keep one reference string per group just for traceability.
agg_dict = {col: 'median' for col in TARGET_COLS}
agg_dict['mol_weight'] = 'median'
agg_dict['Reference'] = lambda refs: '; '.join(sorted(set(refs.dropna().astype(str))))

df = df.groupby(['smiles', 'solvent'], as_index=False).agg(agg_dict)

print(f'Rows before dedup: 20,836  ->  after dedup: {len(df):,}')
print(f'Unique molecules: {df["smiles"].nunique():,}')
print()
print('Non-null counts per target after dedup:')
print(df[TARGET_COLS].notna().sum())

**Cleaning summary:**

- **Duplicates:** 618 rows (~3%) shared a (smiles, solvent) pair with at least one other row, across 304 groups — mostly different papers remeasuring the same compound. Most groups agreed closely; 22 groups disagreed by more than 10 nm on absorption max. Collapsed via median per target, one row per (smiles, solvent) going forward. 20,836 → 20,522 rows; unique molecule count (6,865) is unchanged, since dedup only merges repeated solvent measurements, not distinct molecules.
- **log(extinction coefficient) errors:** already corrected to NaN in Section 5.3 (2 rows).
- **SMILES validity:** already confirmed 100% parseable in Section 4 — no rows dropped on that basis.
- **No other cleaning needed** — absorption max, emission max, and quantum yield all passed their checks in Section 5 as-is.

This `df` is now the cleaned working dataset. `df_raw` still holds the original, untouched 20,836-row file if we ever need to double check something against the source.

Next: featurization — turning each SMILES string into numeric inputs the Random Forest can use, and then the Murcko scaffold split.

## 7. Featurization

Turning each SMILES string into numbers the Random Forest can use. Two kinds of features, computed together:

- **A handful of standard RDKit physicochemical descriptors** (molecular weight, LogP, TPSA, H-bond donors/acceptors, ring counts, etc.) — interpretable, useful later for feature importance.
- **Morgan fingerprint (ECFP, radius 2, 512 bits)** — encodes local substructure patterns, standard choice for this kind of structure-property task.

This is a first-pass feature set to get the pipeline working end to end. Other combinations (descriptors only, bigger fingerprints, different radii, adding solvent information) are worth comparing later — noting that now, not solving it now.

One efficiency point: features only depend on the molecule, not the solvent, so we compute them once per **unique SMILES** (6,865 of them) rather than once per row (20,522) — the same fingerprint would otherwise get recomputed for every solvent a molecule appears with.

In [ ]:
from rdkit.Chem import rdMolDescriptors

# a small, interpretable set of physicochemical descriptors to start with
DESCRIPTOR_FNS = {
    'MolWt': Descriptors.MolWt,
    'MolLogP': Descriptors.MolLogP,
    'TPSA': Descriptors.TPSA,
    'NumHDonors': Descriptors.NumHDonors,
    'NumHAcceptors': Descriptors.NumHAcceptors,
    'NumRotatableBonds': Descriptors.NumRotatableBonds,
    'RingCount': Descriptors.RingCount,
    'NumAromaticRings': Descriptors.NumAromaticRings,
    'FractionCSP3': Descriptors.FractionCSP3,
    'HeavyAtomCount': Descriptors.HeavyAtomCount,
    'NumValenceElectrons': Descriptors.NumValenceElectrons,
}
FP_RADIUS = 2
FP_BITS = 512

unique_smiles = df['smiles'].unique()
print(f'Featurizing {len(unique_smiles):,} unique molecules...')

feature_rows = []
for smi in unique_smiles:
    mol = Chem.MolFromSmiles(smi)  # already know every SMILES here parses, from Section 4
    row = {name: fn(mol) for name, fn in DESCRIPTOR_FNS.items()}
    fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=FP_RADIUS, nBits=FP_BITS)
    row.update({f'fp_{i}': int(bit) for i, bit in enumerate(fp)})
    row['smiles'] = smi
    feature_rows.append(row)

feat_df = pd.DataFrame(feature_rows)
FEATURE_COLS = list(DESCRIPTOR_FNS.keys()) + [f'fp_{i}' for i in range(FP_BITS)]
print(f'Feature matrix: {feat_df.shape[0]:,} molecules x {len(FEATURE_COLS)} features')

In [ ]:
# attach the per-molecule features onto every (molecule, solvent) row
df = df.merge(feat_df, on='smiles', how='left')
df[['smiles', 'solvent'] + list(DESCRIPTOR_FNS.keys())].head()

## 8. Baseline model — random split

Before the scaffold split (the actual point of this project), fit a quick baseline: an ordinary random 80/20 train/test split, default-ish Random Forest, one model per target. This isn't the real evaluation — it's a sanity check that the pipeline works end to end and a reference number to compare the scaffold split against later.

In [ ]:
random_split_results = {}

for target in TARGET_COLS:
    # drop rows missing this particular target - each target has its own usable subset
    sub = df.dropna(subset=[target])
    X = sub[FEATURE_COLS]
    y = sub[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)

    random_split_results[target] = {
        'n_train': len(X_train),
        'n_test': len(X_test),
        'R2': r2_score(y_test, pred),
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': mean_squared_error(y_test, pred) ** 0.5,
    }

pd.DataFrame(random_split_results).T

**Baseline random-split results:**

| Target | n train | n test | R² | MAE | RMSE |
|---|---|---|---|---|---|
| Absorption max | 13,957 | 3,490 | 0.931 | 14.2 nm | 28.1 nm |
| Emission max | 14,852 | 3,714 | 0.845 | 24.4 nm | 36.6 nm |
| log(ext coeff) | 6,551 | 1,638 | 0.826 | 0.139 | 0.239 |
| Quantum yield | 10,998 | 2,750 | 0.546 | 0.145 | 0.206 |

Matches the difficulty ranking from the EDA: wavelength targets easiest, quantum yield clearly the hardest (as expected from the zero-inflated distribution).

**Caveat, directly relevant to the "can we trust this R²" question we raised earlier:** this random split is likely optimistic, for a specific reason — the same molecule appears multiple times under different solvents, so a random split can put, say, compound X in toluene into the training set and compound X in DCM into the test set. Those two rows have an *identical* fingerprint (fingerprints only depend on the molecule, not the solvent), so the model has effectively already seen the test molecule during training. That's not the model generalizing to new chemistry — that's partial leakage through the split. The scaffold split (next section) fixes this by keeping all rows for a given scaffold on one side of the split, and comparing the two numbers is exactly how we'll measure how much of this R² was real vs. leakage.

Next: Murcko scaffold split, and the real comparison.

## 9. Scaffold split — basic check

Same model, same features, same hyperparameters as Section 8. Only thing that changes is how train/test is split: group every molecule by its Murcko scaffold first, then assign whole scaffold groups to train or test — never splitting one scaffold's rows across both sides.

No tuning yet. This is purely to measure the size of the gap against the random-split baseline before deciding whether tuning is even worth the time.

In [ ]:
# get each unique molecule's scaffold (empty string for fully acyclic molecules - no rings to extract)
scaffold_map = {
    smi: MurckoScaffold.MurckoScaffoldSmiles(mol=Chem.MolFromSmiles(smi))
    for smi in df['smiles'].unique()
}
df['scaffold'] = df['smiles'].map(scaffold_map)

print(f"Unique scaffolds: {df['scaffold'].nunique():,} (from {df['smiles'].nunique():,} unique molecules)")
print(f"Rows with no ring system (empty scaffold): {(df['scaffold'] == '').sum()}")
df['scaffold'].value_counts().head()

In [ ]:
# assign whole scaffold groups to train/test, targeting ~80/20 by ROW count (not scaffold count -
# scaffold group sizes vary a lot, e.g. plain benzene ring shows up 400+ times)
scaffold_sizes = df.groupby('scaffold').size().sample(frac=1, random_state=42)  # shuffle group order
cum_rows = scaffold_sizes.cumsum()
test_scaffolds = set(cum_rows[cum_rows <= 0.2 * len(df)].index)

df['split'] = np.where(df['scaffold'].isin(test_scaffolds), 'test', 'train')

print(df['split'].value_counts())
print(f"Distinct scaffolds in train: {df.loc[df['split']=='train','scaffold'].nunique():,}")
print(f"Distinct scaffolds in test:  {df.loc[df['split']=='test','scaffold'].nunique():,}")

overlap = set(df.loc[df['split']=='train','scaffold']) & set(df.loc[df['split']=='test','scaffold'])
print(f"Scaffolds present in both (must be 0): {len(overlap)}")

In [ ]:
scaffold_split_results = {}

for target in TARGET_COLS:
    sub = df.dropna(subset=[target])
    train_sub = sub[sub['split'] == 'train']
    test_sub = sub[sub['split'] == 'test']

    X_train, y_train = train_sub[FEATURE_COLS], train_sub[target]
    X_test, y_test = test_sub[FEATURE_COLS], test_sub[target]

    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)

    scaffold_split_results[target] = {
        'n_train': len(X_train),
        'n_test': len(X_test),
        'R2': r2_score(y_test, pred),
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': mean_squared_error(y_test, pred) ** 0.5,
    }

pd.DataFrame(scaffold_split_results).T

**Scaffold-split results, vs. random-split baseline:**

| Target | R² (random) | R² (scaffold) | Drop | MAE (random) | MAE (scaffold) |
|---|---|---|---|---|---|
| Absorption max | 0.931 | 0.137 | **-0.79** | 14.2 nm | 63.9 nm |
| Emission max | 0.845 | 0.651 | -0.19 | 24.4 nm | 41.0 nm |
| log(ext coeff) | 0.826 | 0.599 | -0.23 | 0.139 | 0.238 |
| Quantum yield | 0.546 | 0.302 | -0.24 | 0.145 | 0.213 |

(3,524 unique scaffolds from 6,865 molecules; split assigned 2,817 scaffolds / 16,426 rows to train and 707 scaffolds / 4,096 rows to test, with zero scaffold overlap between the two confirmed above.)

**This is the headline result of the project so far.** Every target drops when moving from random to scaffold split, which was expected. What wasn't expected: absorption max — the *best*-performing target on the random split — has by far the *worst* collapse on the scaffold split, going from the strongest R² (0.93) to the weakest (0.14). Emission max, log(ext coeff), and quantum yield all drop too, but much more moderately (R² falls by 0.19–0.24, not 0.79).

That's a real, specific hypothesis to chase, not just "scaffold split is harder": absorption max's random-split score looks like it was almost entirely leakage-driven, while the other three targets seem to carry more genuine, scaffold-independent signal. Worth digging into *why* absorption max in particular is so reliant on memorizing near-identical molecules — that's exactly the kind of question Stage 4 (leaf-node inspection) should be able to answer.

Next: wrap this in a proper pipeline and run GridSearchCV (with GroupKFold-by-scaffold) to see how much tuning can recover.

## 10. Hyperparameter tuning — GridSearchCV with scaffold-grouped CV

Tuning the RF properly on the scaffold split, not the random split — that's the honest evaluation, so that's what should drive hyperparameter choice too.

Grid: `n_estimators` ∈ {100, 200, 300}, `max_depth` ∈ {None, 10, 20}, `max_features` ∈ {'sqrt', 0.3}, `min_samples_leaf` ∈ {1, 4} — 36 combinations.

Cross-validation during the search uses `GroupKFold` (5 folds) grouped by scaffold, on the training set only. This matters: plain `KFold` could still split one scaffold's rows across a CV train-fold and val-fold *within* the search, which would make hyperparameter selection a bit optimistic — not fatal to the final test score, but inconsistent with why we're doing a scaffold split in the first place. `GroupKFold` keeps every scaffold fully inside one fold.

One per target, same as everywhere else. This took about 27 minutes total to run (much faster than the ~3.4 hour worst-case estimate from a smaller timing test beforehand — GridSearchCV's parallelism scaled better at full size).

In [ ]:
from sklearn.model_selection import GroupKFold, GridSearchCV

PARAM_GRID = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'max_features': ['sqrt', 0.3],
    'min_samples_leaf': [1, 4],
}

tuned_results = {}

for target in TARGET_COLS:
    sub = df.dropna(subset=[target])
    train_sub = sub[sub['split'] == 'train']
    test_sub = sub[sub['split'] == 'test']

    X_train, y_train = train_sub[FEATURE_COLS], train_sub[target]
    X_test, y_test = test_sub[FEATURE_COLS], test_sub[target]
    groups_train = train_sub['scaffold']

    # materialize the splits as a list - GridSearchCV's parallel workers can't pickle a generator
    gkf = GroupKFold(n_splits=5)
    cv_splits = list(gkf.split(X_train, y_train, groups=groups_train))

    rf = RandomForestRegressor(random_state=42, n_jobs=1)  # n_jobs=1 here, parallelism happens at the search level
    search = GridSearchCV(rf, PARAM_GRID, cv=cv_splits, scoring='r2', n_jobs=-1)
    search.fit(X_train, y_train)

    best_model = search.best_estimator_
    pred = best_model.predict(X_test)

    tuned_results[target] = {
        'n_train': len(X_train),
        'n_test': len(X_test),
        'best_params': search.best_params_,
        'cv_best_score': search.best_score_,  # mean R2 across the 5 scaffold-grouped folds, on training data only
        'test_R2': r2_score(y_test, pred),
        'test_MAE': mean_absolute_error(y_test, pred),
        'test_RMSE': mean_squared_error(y_test, pred) ** 0.5,
    }
    print(f'[{target}] best_params={search.best_params_}  CV R2={search.best_score_:.3f}  test R2={tuned_results[target]["test_R2"]:.3f}')

pd.DataFrame(tuned_results).T

**Tuning results:**

| Target | Best params | CV R² (train, scaffold-grouped) | Test R² (untuned → tuned) | Test MAE (untuned → tuned) |
|---|---|---|---|---|
| Absorption max | n_estimators=300, max_depth=None, max_features=0.3, min_samples_leaf=1 | 0.744 | 0.137 → 0.184 | 63.9 → 60.9 nm |
| Emission max | n_estimators=300, max_depth=None, max_features=0.3, min_samples_leaf=1 | 0.637 | 0.651 → 0.666 | 41.0 → 40.7 nm |
| log(ext coeff) | n_estimators=100, max_depth=None, max_features='sqrt', min_samples_leaf=1 | 0.447 | 0.599 → 0.554 | 0.238 → 0.251 |
| Quantum yield | n_estimators=200, max_depth=None, max_features='sqrt', min_samples_leaf=1 | 0.240 | 0.302 → 0.300 | 0.213 → 0.218 |

**Finding 1 — tuning barely moves the needle.** Best case (absorption max) gains 0.05 R². log(ext coeff) actually got *worse* on the test set despite a full grid search. Quantum yield is flat. None of the 36 hyperparameter combinations manufactured generalization that wasn't already there — which makes sense: tree depth/count/leaf size control how much a model can overfit *the data it's given*, they don't create signal that transfers to chemically novel structures. The scaffold gap is a feature/data limitation, not a tuning problem, and now we have direct evidence of that rather than just an assumption.

**Finding 2 — the real result is the CV-vs-test gap for absorption max specifically.** Its cross-validation score during the search (0.744, computed with scaffold-grouped folds, so this isn't the same leakage as the original random split) is wildly higher than what it actually gets on the held-out scaffold-split test set (0.184). The other three targets don't show this: their CV and test scores are within ~0.15 of each other, in either direction. This says something specific about absorption max — even scaffold-grouped folds *within* the training data aren't a good proxy for how the model performs on the truly held-out test scaffolds. In other words, the training scaffolds are more "mutually predictive" of each other than they are of the test scaffolds, specifically for this target. That's the lead to chase in the leaf-node analysis (Stage 4): what is the model actually keying off for absorption max that doesn't hold up on new structures?

Next: repeat scaffold-split + tuning for the other split variants we discussed (frequency-based scaffold split, solvent-held-out, cluster-based), then move into Stage 4 — inspecting what the best model actually learned.

## 11. Frequency-based scaffold split

A stricter variant of the scaffold split from Section 9. There, scaffold groups were assigned to train/test *randomly*. Here, scaffolds are sorted by how many rows they have, biggest first, and train gets filled with the most common scaffolds until it's ~80% of the data — so test ends up holding only the rarest, most unusual scaffolds. This is the harder generalization test: can the model handle structures that are genuinely uncommon in the dataset, not just "whichever scaffolds the random shuffle happened to hold out."

No tuning yet here either — same untuned RF as Section 9, only the split logic changes, so the comparison is clean.

In [ ]:
# sort scaffolds by group size, biggest first - fill train with the common ones,
# leave only the rare/singleton scaffolds for test
scaffold_sizes_desc = df.groupby('scaffold').size().sort_values(ascending=False)
cum_rows_desc = scaffold_sizes_desc.cumsum()
train_scaffolds_freq = set(cum_rows_desc[cum_rows_desc <= 0.8 * len(df)].index)

df['split_freq'] = np.where(df['scaffold'].isin(train_scaffolds_freq), 'train', 'test')

print(df['split_freq'].value_counts())
print(f"Distinct scaffolds in train: {df.loc[df['split_freq']=='train','scaffold'].nunique():,}")
print(f"Distinct scaffolds in test:  {df.loc[df['split_freq']=='test','scaffold'].nunique():,}")

overlap_freq = set(df.loc[df['split_freq']=='train','scaffold']) & set(df.loc[df['split_freq']=='test','scaffold'])
print(f"Scaffolds present in both (must be 0): {len(overlap_freq)}")

In [ ]:
freq_split_results = {}

for target in TARGET_COLS:
    sub = df.dropna(subset=[target])
    train_sub = sub[sub['split_freq'] == 'train']
    test_sub = sub[sub['split_freq'] == 'test']

    X_train, y_train = train_sub[FEATURE_COLS], train_sub[target]
    X_test, y_test = test_sub[FEATURE_COLS], test_sub[target]

    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)

    freq_split_results[target] = {
        'n_train': len(X_train),
        'n_test': len(X_test),
        'R2': r2_score(y_test, pred),
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': mean_squared_error(y_test, pred) ** 0.5,
    }

pd.DataFrame(freq_split_results).T

**Frequency-split results, vs. the random-scaffold split (Section 9):**

| Target | R² (random-scaffold) | R² (frequency-scaffold) | Direction |
|---|---|---|---|
| Absorption max | 0.137 | **0.584** | up a lot |
| Emission max | 0.651 | 0.548 | down |
| log(ext coeff) | 0.599 | 0.337 | down a lot |
| Quantum yield | 0.302 | 0.102 | down a lot |

(3,524 scaffolds; frequency split puts the 1,188 most common scaffolds / 16,416 rows in train and the 2,336 rarest scaffolds / 4,106 rows in test, zero overlap confirmed.)

**This flips the Section 9 story for absorption max, and that's important.** Under the frequency split — which forces every common, well-represented scaffold into training — absorption max's R² jumps from 0.137 to 0.584, better than three of the four targets. That strongly suggests the catastrophic 0.137 score in Section 9 wasn't really "absorption max generalizes badly" — it was more likely **bad luck in which scaffolds the random shuffle happened to assign to test**. If one or two large, structurally unusual scaffold groups randomly landed in the Section 9 test set, that alone could tank the score, since R² for a correlation-like metric is sensitive to a handful of badly-predicted points pulling the whole thing down.

**Meanwhile the other three targets get *harder*, not easier, under the frequency split** — the opposite direction from absorption max. Emission max drops modestly (0.651 → 0.548), but log(ext coeff) and quantum yield both fall sharply. This makes sense for a different reason: those two targets already have the least data (8,189 and 13,748 rows respectively), so forcing all the common/easy scaffolds into training and leaving only rare, singleton-like scaffolds for test is a genuinely harder ask when there isn't much nearby structure in training to lean on.

**This changes the plan slightly.** A single random scaffold split can apparently swing a target's measured R² by 0.4+ just from which scaffolds happened to land in test — that's the scaffold k-fold CV idea from earlier, now justified by direct evidence rather than just caution. Before trusting any single split's number (ours or the frequency one), we should run scaffold-grouped k-fold CV and look at the *distribution* of scores across folds, not a single train/test number. Proposing we do that next, before solvent-held-out and cluster-based splits — it's more informative per unit of effort given what we just found.

## 12. Hypothesis test — was absorption max's collapse real, or an unlucky split?

**The hypothesis:** absorption max's catastrophic score in Section 9 (R² 0.137) was not a genuine sign that the model fails to generalize to new scaffolds for this target. It was an artifact of *which specific scaffolds* the random shuffle happened to assign to the test set — a small number of unusual, hard-to-predict scaffold groups landing in test by chance, dragging R² down. If that's right, a *different* random scaffold split should give a very different score, and the swing we already saw between Section 9 (0.137) and the frequency split (0.584) is exactly that kind of evidence, just from two single data points.

**How we're testing it:** scaffold-grouped 5-fold cross-validation (`GroupKFold`, grouped by scaffold) across the *entire* dataset for all four targets — not just absorption max. Each fold trains on 4/5 of the scaffolds and tests on the held-out 1/5, so we get 5 independent "what if this particular set of scaffolds had been the test set" measurements per target. If the hypothesis is right, absorption max's 5 scores should vary a lot more than the other three targets', and its average should land well above 0.137. Untuned RF (same `n_estimators=200` as Sections 9 and 11), so this isolates split variance specifically, not tuning effects.

In [ ]:
kfold_r2 = {}

for target in TARGET_COLS:
    sub = df.dropna(subset=[target]).reset_index(drop=True)
    X, y, groups = sub[FEATURE_COLS], sub[target], sub['scaffold']

    gkf = GroupKFold(n_splits=5)
    fold_scores = []
    for train_idx, test_idx in gkf.split(X, y, groups=groups):
        rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
        rf.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = rf.predict(X.iloc[test_idx])
        fold_scores.append(r2_score(y.iloc[test_idx], pred))

    kfold_r2[target] = fold_scores
    print(f'{target}: {[round(s, 3) for s in fold_scores]}  '
          f'mean={np.mean(fold_scores):.3f}  std={np.std(fold_scores):.3f}')

**Results:**

| Target | Fold scores | Mean R² | Std R² |
|---|---|---|---|
| Absorption max | 0.713, 0.334, 0.725, 0.706, 0.729 | **0.641** | **0.154** |
| Emission max | 0.629, 0.618, 0.616, 0.643, 0.625 | 0.626 | 0.010 |
| log(ext coeff) | 0.508, 0.392, 0.541, 0.514, 0.312 | 0.454 | 0.088 |
| Quantum yield | 0.265, 0.268, 0.183, 0.297, 0.195 | 0.242 | 0.044 |

**Hypothesis confirmed.** Absorption max's fold-to-fold standard deviation (0.154) is 1.75x–15x larger than every other target's. Four of its five folds land in a tight 0.706–0.729 band; the fifth (0.334) is the odd one out — and even that unlucky fold is more than double the original Section 9 score of 0.137. So Section 9's single random split wasn't just somewhat unlucky, it was an extreme draw, worse than any of these five folds. Absorption max's *real* typical scaffold-split performance looks to be around R² 0.64 — right in line with emission max, not the clear worst performer we thought it was two sections ago.

**This also reframes emission max as the most reliable target of the four**, not just a "moderate" one — its std (0.010) is dramatically tighter than the others, meaning its ~0.62-0.65 R² holds up almost regardless of which scaffolds end up in test. log(ext coeff) and quantum yield sit in between: moderate variance, consistent with having less data to begin with.

**Methodological takeaway for the rest of this project:** a single scaffold train/test split is not a reliable number on its own, at least not without checking variance first. From here on, scaffold-grouped k-fold CV (not a single split) should be the default way we report any scaffold-based result — including when we get to comparing split types (solvent-held-out, cluster-based) and picking the "best" model in Stage 4. The earlier single-split numbers in Sections 9–11 aren't wrong exactly, but they should be read as one sample from a distribution, not as the answer.

Next: solvent-held-out split, this time using k-fold from the start rather than a single split.

## 13. Adding solvent features (prerequisite for a solvent-held-out split)

Before a solvent-held-out split can mean anything, the model needs to be able to tell solvents apart in the first place. Right now `FEATURE_COLS` is purely molecule-based (RDKit descriptors + fingerprint computed from the chromophore's SMILES) — the model has no way to predict differently for the same molecule in two different solvents, since nothing in its input changes.

Good news: the `Solvent` column in this dataset is itself SMILES (`O` = water, `ClCCl` = DCM, `Cc1ccccc1` = toluene, etc.), so we can run the same style of RDKit featurization on it that we already built for the chromophore. A small, chemically-motivated set of descriptors for *polarity* (since that's the main thing that drives solvent effects on photophysics): molecular weight, LogP (polarity/lipophilicity), TPSA, and H-bond donor/acceptor counts.

One exception: 20 rows have `Solvent = 'gas'`, which isn't a real SMILES (these are the gas-phase, far-UV absorbers we found back in Section 5.1's outlier check). Giving those an explicit `is_gas_phase` flag and zeroing out the other solvent descriptors, rather than dropping the rows or letting them fail to parse.

This creates a second feature set (`FEATURE_COLS_SOLV`) used only from here onward. Sections 8–12 above keep using the original molecule-only `FEATURE_COLS`, so none of those earlier results change retroactively.

In [ ]:
SOLVENT_DESCRIPTOR_FNS = {
    'solvent_MolWt': Descriptors.MolWt,
    'solvent_MolLogP': Descriptors.MolLogP,
    'solvent_TPSA': Descriptors.TPSA,
    'solvent_NumHDonors': Descriptors.NumHDonors,
    'solvent_NumHAcceptors': Descriptors.NumHAcceptors,
}

unique_solvents = df['solvent'].unique()
solvent_rows = []
for solv in unique_solvents:
    mol = Chem.MolFromSmiles(solv)
    if mol is None:
        # 'gas' - not a real SMILES, represents gas-phase (no solvent) measurements
        row = {name: 0.0 for name in SOLVENT_DESCRIPTOR_FNS}
        row['solvent_is_gas_phase'] = 1
    else:
        row = {name: fn(mol) for name, fn in SOLVENT_DESCRIPTOR_FNS.items()}
        row['solvent_is_gas_phase'] = 0
    row['solvent'] = solv
    solvent_rows.append(row)

solvent_feat_df = pd.DataFrame(solvent_rows)
SOLVENT_FEATURE_COLS = list(SOLVENT_DESCRIPTOR_FNS.keys()) + ['solvent_is_gas_phase']

df = df.merge(solvent_feat_df, on='solvent', how='left')
FEATURE_COLS_SOLV = FEATURE_COLS + SOLVENT_FEATURE_COLS

print(f'Solvent features computed for {len(unique_solvents):,} unique solvents')
print(f'Molecule-only features: {len(FEATURE_COLS)}  ->  with solvent features: {len(FEATURE_COLS_SOLV)}')
df[['smiles', 'solvent'] + SOLVENT_FEATURE_COLS].drop_duplicates('solvent').head()

## 14. Solvent-held-out split

Tests a different kind of generalization than the scaffold splits: not "new molecule," but "new solvent environment." Every row for a given solvent goes entirely to one side — train or test — same grouping logic as the scaffold split, just grouped by `solvent` instead of `scaffold`. The same molecule *can* (and usually does) appear on both sides here, in different solvents — that's intentional, not leakage, since `FEATURE_COLS_SOLV` now includes solvent descriptors, so the model has to actually use the solvent information to do well, rather than just recalling the molecule.

Following the methodological rule from Section 12: going straight to **solvent**-grouped 5-fold CV rather than a single split, since we now know a single split's number can be misleading.

In [ ]:
kfold_solvent_r2 = {}

for target in TARGET_COLS:
    sub = df.dropna(subset=[target]).reset_index(drop=True)
    X, y, groups = sub[FEATURE_COLS_SOLV], sub[target], sub['solvent']

    gkf = GroupKFold(n_splits=5)
    fold_scores = []
    for train_idx, test_idx in gkf.split(X, y, groups=groups):
        rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
        rf.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = rf.predict(X.iloc[test_idx])
        fold_scores.append(r2_score(y.iloc[test_idx], pred))

    kfold_solvent_r2[target] = fold_scores
    print(f'{target}: {[round(s, 3) for s in fold_scores]}  '
          f'mean={np.mean(fold_scores):.3f}  std={np.std(fold_scores):.3f}')

**Results, vs. the scaffold-split k-fold means from Section 12:**

| Target | Fold scores (solvent-held-out) | Mean R² | Std R² | Mean R² (scaffold, Sec. 12) |
|---|---|---|---|---|
| Absorption max | 0.822, 0.940, 0.849, 0.934, 0.905 | **0.890** | 0.047 | 0.641 |
| Emission max | 0.755, 0.872, 0.783, 0.860, 0.850 | 0.824 | 0.046 | 0.626 |
| log(ext coeff) | 0.437, 0.791, 0.701, 0.783, 0.566 | 0.656 | 0.136 | 0.454 |
| Quantum yield | 0.448, 0.525, 0.546, 0.592, 0.584 | 0.539 | 0.052 | 0.242 |

(1,363 unique solvents; dichloromethane, acetonitrile, and toluene are the three largest groups.)

**Every target generalizes to new solvents noticeably better than to new scaffolds.** That's a genuinely informative result on its own, not just "another split to check off the list": it says the dominant source of difficulty in this whole problem is **chemical structure**, not **solvent environment**. The model has an easier time predicting a familiar molecule's behavior in a solvent it's never trained on than predicting a never-seen molecule's behavior at all — which makes physical sense. Solvent effects on absorption/emission are real but comparatively smooth, second-order shifts (a few nm to a few tens of nm); switching to an unfamiliar *molecular scaffold* changes the underlying electronic structure far more fundamentally.

Quantum yield is the one target where even the "easier" solvent-held-out task still tops out around R² 0.54 — consistent with the zero-inflation issue we identified back in Section 5.4 being a problem independent of which axis (molecule or solvent) we're generalizing across.

**For Stage 4:** this is a useful contrast to keep in mind when inspecting the "best" model later — a model that looks strong might be strong specifically at solvent interpolation while still being weak at structural extrapolation, and those are different kinds of "good." Worth checking which one the feature importances actually reflect.

Next: cluster-based split (Butina), then into Stage 4 — inspecting what the best model has actually learned.

## 15. H2 — do chemistry-rule features generalize better than fingerprint bits?

**The hypothesis (from an external review of this project's approach):** a Morgan fingerprint encodes exact local substructure patterns — "have I seen this specific fragment shape before." That's great for memorizing structures the model has already seen, but a genuinely new scaffold produces mostly unfamiliar bit patterns, with no principled fallback. A feature built from an actual physical mechanism (e.g. "does this molecule have a conjugated push-pull donor-acceptor system") describes a *concept* that should still apply to a structure the model has never seen, since the mechanism itself doesn't depend on having encountered that exact scaffold before. If that's right, a small set of mechanism-grounded features should generalize across the scaffold split at least as well as — maybe better than — 512 fingerprint bits.

**The rule features (20 new ones, on top of the 11 descriptors we already had):**
- **Donor subtypes:** primary/secondary/tertiary amine, phenolic -OH, aryl ether/alkoxy (electron-donating groups → push electron density into the ring → red shift)
- **Acceptor subtypes:** nitrile, nitro, carbonyl, CF3, sulfonyl (electron-withdrawing groups → same mechanism, opposite end)
- **`tict_rotor`:** dialkylamino group on an aromatic ring via a rotatable bond — a flexible donor that can twist in the excited state and open a non-radiative decay path (lowers Φ)
- **`halogen_heavy` / `halogen_any`:** heavy-atom effect → boosts intersystem crossing → lowers Φ
- **`donor_total`, `acceptor_total`, `has_push_pull`, `donor_acceptor_product`:** aggregated/interaction versions of the above
- **`conjugation_size`:** size of the largest connected conjugated chunk (walking the bond graph via union-find over `bond.GetIsConjugated()`) — longer conjugation → smaller energy gap → red shift
- **`formal_charge`, `ring_fraction`:** ionic character and rigidity (rigid structures lose less energy to vibration → higher Φ)

Every SMARTS pattern was sanity-checked against known reference molecules before trusting it at scale — e.g. DMABN (the textbook TICT dye) correctly triggers the donor, acceptor, *and* TICT-rotor flags simultaneously.

**The test:** three feature sets, same scaffold-grouped 5-fold CV as Section 12, same untuned RF, across all four targets:
- `fingerprint_only` — the original 523 features (11 descriptors + 512 fingerprint bits)
- `rule_only` — the 11 descriptors + the 20 new chemistry-rule features, **no fingerprint at all** (31 features total)
- `combined` — all of the above together (543 features)

If rule-only matches or beats fingerprint-only, that's strong support for the hypothesis. If combined beats fingerprint-only even by a little, that shows the rule features add something fingerprint was missing, even while outnumbered 512-to-20.

In [ ]:
SMARTS_PATTERNS = {
    'donor_NH2': '[NX3H2][#6]',                       # primary amine
    'donor_NH1': '[NX3H1]([#6])[#6]',                 # secondary amine
    'donor_NH0': '[NX3H0]([#6])([#6])[#6]',           # tertiary amine
    'donor_OH_arom': '[OX2H][c]',                      # phenol
    'donor_OR_arom': '[OX2]([#6])[c]',                 # aryl ether/alkoxy
    'acceptor_nitrile': 'C#N',
    'acceptor_nitro': '[N+](=O)[O-]',
    'acceptor_carbonyl': '[CX3]=[OX1]',
    'acceptor_CF3': 'C(F)(F)F',
    'acceptor_sulfonyl': 'S(=O)(=O)',
    'tict_rotor': '[NX3]([#6])([#6])!@[c]',           # dialkylamino on aromatic, rotatable bond
    'halogen_heavy': '[Br,I]',
    'halogen_any': '[F,Cl,Br,I]',
}
COMPILED_SMARTS = {name: Chem.MolFromSmarts(s) for name, s in SMARTS_PATTERNS.items()}
assert all(p is not None for p in COMPILED_SMARTS.values()), 'a SMARTS pattern failed to compile'

# sanity check against known reference molecules before trusting these at scale
test_cases = {
    'DMABN (classic TICT dye)': 'CN(C)c1ccc(C#N)cc1',   # should hit donor_NH0, acceptor_nitrile, tict_rotor
    'nitrobenzene': 'c1ccccc1[N+](=O)[O-]',
    'phenol': 'c1ccccc1O',
    'bromobenzene': 'c1ccccc1Br',
}
for label, smi in test_cases.items():
    mol = Chem.MolFromSmiles(smi)
    hits = {name: len(mol.GetSubstructMatches(patt)) for name, patt in COMPILED_SMARTS.items()
            if len(mol.GetSubstructMatches(patt)) > 0}
    print(f'{label}: {hits}')

In [ ]:
def largest_conjugated_system(mol):
    # union-find over atoms connected by a conjugated bond; return the size of the biggest cluster
    n = mol.GetNumAtoms()
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    for bond in mol.GetBonds():
        if bond.GetIsConjugated():
            union(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())
    sizes = {}
    for atom in mol.GetAtoms():
        root = find(atom.GetIdx())
        sizes[root] = sizes.get(root, 0) + 1
    return max(sizes.values()) if sizes else 0


rule_rows = []
for smi in unique_smiles:
    mol = Chem.MolFromSmiles(smi)
    smarts_counts = {name: len(mol.GetSubstructMatches(patt)) for name, patt in COMPILED_SMARTS.items()}

    row = dict(smarts_counts)
    row['donor_total'] = sum(smarts_counts[k] for k in ['donor_NH2', 'donor_NH1', 'donor_NH0', 'donor_OH_arom', 'donor_OR_arom'])
    row['acceptor_total'] = sum(smarts_counts[k] for k in ['acceptor_nitrile', 'acceptor_nitro', 'acceptor_carbonyl', 'acceptor_CF3', 'acceptor_sulfonyl'])
    row['has_push_pull'] = int(row['donor_total'] > 0 and row['acceptor_total'] > 0)
    row['donor_acceptor_product'] = row['donor_total'] * row['acceptor_total']
    row['conjugation_size'] = largest_conjugated_system(mol)
    row['formal_charge'] = Chem.GetFormalCharge(mol)
    n_ring_atoms = sum(1 for atom in mol.GetAtoms() if atom.IsInRing())
    row['ring_fraction'] = n_ring_atoms / mol.GetNumHeavyAtoms() if mol.GetNumHeavyAtoms() > 0 else 0.0
    row['smiles'] = smi
    rule_rows.append(row)

rule_feat_df = pd.DataFrame(rule_rows)
df = df.merge(rule_feat_df, on='smiles', how='left')

RULE_FEATURE_COLS = (list(DESCRIPTOR_FNS.keys()) + list(SMARTS_PATTERNS.keys()) +
                      ['donor_total', 'acceptor_total', 'has_push_pull', 'donor_acceptor_product',
                       'conjugation_size', 'formal_charge', 'ring_fraction'])

FEATURE_SETS = {
    'fingerprint_only': FEATURE_COLS,
    'rule_only': RULE_FEATURE_COLS,
    'combined': FEATURE_COLS + [c for c in RULE_FEATURE_COLS if c not in DESCRIPTOR_FNS],
}
print('Feature set sizes:', {k: len(v) for k, v in FEATURE_SETS.items()})

In [ ]:
h2_results = {}

for target in TARGET_COLS:
    sub = df.dropna(subset=[target]).reset_index(drop=True)
    groups = sub['scaffold']
    h2_results[target] = {}

    for set_name, cols in FEATURE_SETS.items():
        X, y = sub[cols], sub[target]
        gkf = GroupKFold(n_splits=5)
        fold_scores = []
        importances_accum = None

        for train_idx, test_idx in gkf.split(X, y, groups=groups):
            rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
            rf.fit(X.iloc[train_idx], y.iloc[train_idx])
            pred = rf.predict(X.iloc[test_idx])
            fold_scores.append(r2_score(y.iloc[test_idx], pred))
            importances_accum = rf.feature_importances_.copy() if importances_accum is None else importances_accum + rf.feature_importances_

        mean_importances = importances_accum / 5
        top_features = sorted(zip(cols, mean_importances), key=lambda t: -t[1])[:6]

        h2_results[target][set_name] = {
            'mean_r2': np.mean(fold_scores),
            'std_r2': np.std(fold_scores),
            'n_features': len(cols),
            'top_features': top_features,
        }
        print(f'[{target} / {set_name}] mean R2={np.mean(fold_scores):.3f} std={np.std(fold_scores):.3f} '
              f'(n_features={len(cols)})')

**Results:**

| Target | fingerprint_only (523 feat) | rule_only (31 feat) | combined (543 feat) |
|---|---|---|---|
| Absorption max | 0.641 | 0.481 | **0.659** |
| Emission max | 0.626 | 0.527 | **0.650** |
| log(ext coeff) | 0.454 | 0.204 | **0.476** |
| Quantum yield | 0.242 | 0.093 | **0.244** |

**The strong version of the hypothesis doesn't hold.** Rule-only loses to fingerprint-only on every single target — sometimes by a lot (log(ext coeff): 0.204 vs. 0.454). 31 mechanism-grounded features, on their own, don't carry enough raw signal to replace 512 fingerprint bits. The "generalizable concept beats memorized fragment" story is too simple as stated.

**But the combined set beats fingerprint-only every time, too** — modestly (0.018-0.022 R² for three targets, essentially flat for quantum yield), but consistently in the same direction. That's the real, defensible finding: the rule features aren't a replacement for the fingerprint, but they add *something* it was missing, even while outnumbered 512-to-20 at every tree split. That's a meaningfully different, more honest conclusion than the original hypothesis — not "physics beats memorization," but "physics adds a small, consistent amount of signal on top of memorization."

**The standout feature is `conjugation_size`.** Across almost every target and feature set, it ranks in the top 2-3 most important features — frequently beating every individual fingerprint bit except one dominant one (`fp_463`, which shows up as the single most important feature for absorption max, emission max, and log(ext coeff)). This is a genuine, independent validation of basic photophysics theory (longer conjugation → smaller HOMO-LUMO gap → red-shifted absorption/emission) showing up directly in what the model actually uses, not just assumed going in.

**Other interpretable patterns in the feature importances:**
- `ring_fraction`, `MolLogP`, and `TPSA` (rigidity and polarity) dominate the rule-only feature list for quantum yield — directly matching the "rigid structures lose less energy to vibration" and solvent-polarity mechanisms discussed.
- `donor_NH0` (tertiary amine donor) ranks highly specifically for *emission* max, not absorption — consistent with TICT being primarily an excited-state (i.e. emission-side) phenomenon.
- `halogen_any` shows up for absorption max — plausible, though less mechanistically specific than the others.

**Lead for Stage 4:** `fp_463` is doing an outsized amount of work across three of four targets. RDKit can decode exactly which substructure environment a specific Morgan bit corresponds to (via the fingerprint's bit-info dictionary) — worth checking what that bit actually represents when we get to leaf-node inspection, since a single bit dominating this much is either a real, important structural motif, or a sign the model is leaning on something narrower and more memorization-prone than it should.

Next: cluster-based split (Butina), then Stage 4.